# trafficnews — 데이터 정제·검수·학습 데이터 준비

기존 `trafficnews_clean.py`, `trafficnews_normalize.py`, `trafficnews_verify.py`, `trafficnews_finalize.py`, `trafficnews_make_pairs.py`에서 진행한 작업을 노트북에서 단계별로 재현합니다.

- **기존 `.py`, CSV, PDF 파일을 수정하거나 덮어쓰지 않습니다.**
- 모든 작업은 메모리에서 진행하고 결과를 출력해 확인합니다.
- 프로젝트 폴더 `C:\workspace\trafficnews`에서 열고 `.venv` Python 커널을 선택하세요.
- 우선 **1단계 코드 셀만 실행**하고 결과를 확인하세요. 나머지는 이후 단계에서 차례로 진행합니다.


## 1단계. 원본 CSV 읽기 (지금 실행할 셀)

In [1]:
from pathlib import Path
import pandas as pd

# 이 노트북은 프로젝트 폴더에서 실행합니다.
project_dir = Path.cwd()
source_file = project_dir / "trafficnews_crawling.csv"

print("현재 작업 폴더:", project_dir)
print("원본 CSV 존재:", source_file.exists())

if not source_file.exists():
    print("CSV를 찾지 못했습니다. 노트북의 작업 폴더를 확인하세요.")
else:
    original_df = pd.read_csv(source_file)
    print("데이터 행 수:", len(original_df))
    print("열 이름:", original_df.columns.tolist())
    if "본문" in original_df.columns and len(original_df) > 0:
        original_text = str(original_df.loc[0, "본문"])
        print("첫 번째 본문 글자 수:", len(original_text))
        print("첫 번째 본문 앞부분:", original_text[:120])

현재 작업 폴더: c:\workspace\trafficnews
원본 CSV 존재: True
데이터 행 수: 1
열 이름: ['제목', '담당부서', '등록일', '원본URL', 'PDF_URL', '본문']
첫 번째 본문 글자 수: 1707
첫 번째 본문 앞부분: - 1 -
보도자료
보도시점 : 2026. 11. 10.(화) 11:00 이후(11. 11.(수) 조간) / 배포 : 2026. 11. 10.(화)
신도시 교통대책 속도 높인다
… 도로사업 16곳 집중 점검
- 대광


## 2단계. 원본 본문 정제 (1단계 확인 후 실행)

In [2]:
import re


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # PDF에서 추출된 페이지 번호 제거
    text = re.sub(r"(?m)^\s*-\s*\d+\s*-\s*$", "", text)
    # 이전 실습에서 확인한 문서 끝 연락처 영역 제외
    footer = "\n담당부서\n대도시권광역교통위원회"
    if footer in text:
        text = text.split(footer, 1)[0]
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)

cleaned_df = original_df.copy()
cleaned_df["본문"] = cleaned_df["본문"].apply(clean_text)
print("정제 후 첫 본문 글자 수:", len(cleaned_df.loc[0, "본문"]))
print("담당자 전화번호 포함:", bool(re.search(r"044-\d{3}-\d{4}", cleaned_df.loc[0, "본문"])))

정제 후 첫 본문 글자 수: 1558
담당자 전화번호 포함: False


## 3단계. 줄바꿈 정리 (2단계 확인 후 실행)

In [4]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

normalized_df = cleaned_df.copy()
normalized_df["본문"] = normalized_df["본문"].apply(normalize_text)
print("줄바꿈 개수:", normalized_df.loc[0, "본문"].count("\n"))
print("정리 후 본문 글자 수:", len(normalized_df.loc[0, "본문"]))

줄바꿈 개수: 0
정리 후 본문 글자 수: 1558


## 4단계. 원본 PDF와 대조한 띄어쓰기 수정 (3단계 확인 후 실행)

In [5]:
corrections = {
    "확장및 통합": "확장 및 통합",
    "인허가등": "인허가 등",
    "특히,TF": "특히, TF",
}

final_df = normalized_df.copy()
for before, after in corrections.items():
    final_df["본문"] = final_df["본문"].str.replace(before, after, regex=False)

print("수정 후 본문 글자 수:", len(final_df.loc[0, "본문"]))
for corrected in corrections.values():
    print(corrected, ":", "OK" if corrected in final_df.loc[0, "본문"] else "확인 필요")

수정 후 본문 글자 수: 1558
확장 및 통합 : OK
인허가 등 : OK
특히, TF : OK


## 5단계. 핵심 표현 검증 (4단계 확인 후 실행)

이 검사는 핵심 표현의 존재만 확인합니다. 전체 본문과 원본 PDF의 완전 일치를 증명하지 않습니다.

In [6]:
check_items = [
    "16개 도로사업", "11월 11일", "10월 26일", "2개 사업",
    "하남교산", "인천계양", "과천과천", "안산장상", "시흥장현",
]
original_text = str(original_df.loc[0, "본문"])
final_text = str(final_df.loc[0, "본문"])

for phrase in check_items:
    ok = phrase in original_text and phrase in final_text
    print(f"{phrase}: {'OK' if ok else '확인 필요'}")

16개 도로사업: OK
11월 11일: OK
10월 26일: OK
2개 사업: OK
하남교산: OK
인천계양: OK
과천과천: OK
안산장상: OK
시흥장현: OK


## 6단계. 기존 최종 CSV와 결과 비교 (5단계 확인 후 실행)

실제 파일은 수정하지 않고 기존 저장 결과와 비교만 합니다.

In [7]:
existing_final = project_dir / "trafficnews_final.csv"
if not existing_final.exists():
    print("기존 최종 CSV가 없습니다. 비교를 건너뜁니다.")
else:
    existing_df = pd.read_csv(existing_final)
    same_text = (len(existing_df) == len(final_df) and
                 str(existing_df.loc[0, "본문"]) == str(final_df.loc[0, "본문"]))
    print("기존 최종 CSV와 첫 본문 완전 일치:", same_text)
    if not same_text:
        print("차이가 있습니다. 원본과 기존 .py 처리 결과를 대조해야 합니다.")

기존 최종 CSV와 첫 본문 완전 일치: True


## 7단계. 본문–리드 학습 데이터 쌍 확인 (6단계 확인 후 실행)

이전에 확정한 리드를 사용하며 이 셀은 CSV를 저장하지 않습니다.

In [8]:
lead = (
    "국토교통부 대도시권광역교통위원회가 2026년 11월 11일 "
    "관계기관 합동 TF 회의를 열어 신도시 교통대책에 포함된 "
    "도로사업 16곳의 추진 상황과 인허가 지연 요인을 "
    "점검할 계획이라고 밝혔다."
)
pairs_df = pd.DataFrame([{
    "본문": final_df.loc[0, "본문"],
    "리드": lead,
    "판정": "사용",
    "메모": "원본 PDF와 핵심 내용 대조",
    "원본URL": final_df.loc[0, "원본URL"],
}])
print("학습 데이터 쌍:", len(pairs_df))
print("리드:", pairs_df.loc[0, "리드"])
print("기존 파일은 덮어쓰지 않았습니다.")

학습 데이터 쌍: 1
리드: 국토교통부 대도시권광역교통위원회가 2026년 11월 11일 관계기관 합동 TF 회의를 열어 신도시 교통대책에 포함된 도로사업 16곳의 추진 상황과 인허가 지연 요인을 점검할 계획이라고 밝혔다.
기존 파일은 덮어쓰지 않았습니다.


In [10]:
import pandas as pd

# 전체 원본 보도자료 19건
raw_df = pd.read_csv("trafficnews_all_19_raw.csv")

# 기존에 정제한 보도자료 10건
cleaned_10_df = pd.read_csv("trafficnews_all_10_cleaned.csv")

# 기존 정제 자료의 ID 목록
cleaned_ids = set(cleaned_10_df["보도자료ID"].astype(str))

# 아직 정제하지 않은 보도자료만 선택
new_df = raw_df[
    ~raw_df["보도자료ID"].astype(str).isin(cleaned_ids)
].copy()

# 본문 상태 검사
new_df["본문글자수"] = new_df["본문"].str.len()
new_df["본문누락"] = new_df["본문"].isna() | (
    new_df["본문"].fillna("").str.strip() == ""
)

# 결과 출력
print("===== STEP 5-1 정제 대상 확인 =====")
print("전체 원본:", len(raw_df))
print("기존 정제 자료:", len(cleaned_10_df))
print("추가 정제 대상:", len(new_df))
print("본문 누락:", new_df["본문누락"].sum())
print("중복 ID:", new_df["보도자료ID"].duplicated().sum())

display(
    new_df[
        ["보도자료ID", "제목", "페이지수", "본문글자수", "본문누락"]
    ]
)

===== STEP 5-1 정제 대상 확인 =====
전체 원본: 19
기존 정제 자료: 10
추가 정제 대상: 9
본문 누락: 0
중복 ID: 0


,보도자료ID,제목,페이지수,본문글자수,본문누락
10,95092504,모두의카드 카드정보 현행화하지 않은 이용자는 10월 26일까지 현행화하세요!,2,1622,False
11,95092486,"택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다",2,1205,False
12,95092459,모두의카드 출퇴근 시차시간 혜택 계속 이어간다,3,2515,False
13,95092458,[장관동정] 추석 귀성길 안전·편의 빈틈없도록 … 주요 교통현장 집중 점검,2,1774,False
14,95092447,APEC 교통장관회의서 AI·스마트 모빌리티 미래협력 제안 나선다,5,4158,False
15,95092445,"국토부·기후부, 두 마리 토끼 잡는다... 모두의카드·그린카드 연계",3,2009,False
16,95092439,추석 연휴 교통혼잡 해소에 총력... 정부 합동 특별교통대책을 시행합니다,61,48092,False
17,95092438,안전운전 배달 종사자 보험료 최대 11% 할인,3,2047,False
18,95092430,5극3특 국가균형발전 뒷받침할 초광역교통 전략 연내 마련,2,1497,False


In [11]:

import re
import pandas as pd

def prepare_body(text):
    # PDF 페이지 번호 제거
    text = re.sub(r"(?m)^[ \t]*-[ \t]*\d+[ \t]*-[ \t]*$", "", text)

    lines = text.splitlines()

    # 담당자 정보 영역의 시작 위치 탐색
    contact_positions = [
        i for i, line in enumerate(lines)
        if re.fullmatch(r"\s*담당\s*부서\s*", line)
    ]

    # 경계가 명확하지 않으면 원문 전체 보존
    if len(contact_positions) != 1:
        return text.strip(), "경계 검토 필요"

    start = contact_positions[0]

    # 담당부서 앞부분을 학습용 본문 후보로 분리
    article_lines = lines[:start]

    while article_lines and article_lines[-1].strip() in {"", "-", "`"}:
        article_lines.pop()

    return "\n".join(article_lines).strip(), "검수 필요"

# 기존 데이터는 유지하고 새로운 열에 결과 저장
new_cleaned_df = new_df.copy()

results = new_cleaned_df["본문"].apply(prepare_body)

new_cleaned_df["학습본문"] = results.apply(lambda x: x[0])
new_cleaned_df["검수상태"] = results.apply(lambda x: x[1])
new_cleaned_df["학습본문글자수"] = new_cleaned_df["학습본문"].str.len()

# 전화번호 잔존 여부 확인
phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

new_cleaned_df["전화번호포함"] = new_cleaned_df["학습본문"].apply(
    lambda text: bool(re.search(phone_pattern, text))
)

display(new_cleaned_df[
    ["보도자료ID", "학습본문글자수", "검수상태", "전화번호포함"]
])

print("\n전체:", len(new_cleaned_df))
print("경계 검토 필요:", (new_cleaned_df["검수상태"] == "경계 검토 필요").sum())
print("전화번호 포함:", new_cleaned_df["전화번호포함"].sum())


,보도자료ID,학습본문글자수,검수상태,전화번호포함
10,95092504,1509,검수 필요,False
11,95092486,1204,경계 검토 필요,True
12,95092459,2396,검수 필요,False
13,95092458,1409,검수 필요,False
14,95092447,1574,검수 필요,False
15,95092445,1629,검수 필요,False
16,95092439,47746,경계 검토 필요,True
17,95092438,1932,검수 필요,False
18,95092430,1376,검수 필요,False



전체: 9
경계 검토 필요: 2
전화번호 포함: 2


In [12]:
import re
import pandas as pd

review_rows = []

for news_id in [95092486, 95092439]:
    row = new_df.loc[
        new_df["보도자료ID"].astype(str) == str(news_id)
    ].iloc[0]

    lines = row["본문"].splitlines()

    for index, line in enumerate(lines):
        if re.search(
            r"담당\s*부서|책임자|담당자|0\d{1,2}[- ]\d{3,4}[- ]\d{4}",
            line
        ):
            review_rows.append({
                "보도자료ID": news_id,
                "줄번호": index + 1,
                "내용": line,
            })

review_df = pd.DataFrame(review_rows)

display(review_df)

,보도자료ID,줄번호,내용
0,95092486,24,담당 부서
1,95092486,26,책임자
2,95092486,28,한진선(044-202-7991)
3,95092486,30,담당자
4,95092486,32,정혜진(044-202-7616)
...,...,...,...
74,95092439,4362,054-880-8242
75,95092439,4364,054-850-3292
76,95092439,4367,055-211-5753
77,95092439,4369,055-254-4173


In [13]:
import re
import pandas as pd

review_rows = []

for news_id in [95092486, 95092439]:
    row = new_df.loc[
        new_df["보도자료ID"].astype(str) == str(news_id)
    ].iloc[0]

    lines = row["본문"].splitlines()

    for index, line in enumerate(lines):
        if re.search(
            r"담당\s*부서|책임자|담당자|0\d{1,2}[- ]\d{3,4}[- ]\d{4}",
            line
        ):
            review_rows.append({
                "보도자료ID": news_id,
                "줄번호": index + 1,
                "내용": line,
            })

review_df = pd.DataFrame(review_rows)

display(review_df)

,보도자료ID,줄번호,내용
0,95092486,24,담당 부서
1,95092486,26,책임자
2,95092486,28,한진선(044-202-7991)
3,95092486,30,담당자
4,95092486,32,정혜진(044-202-7616)
...,...,...,...
74,95092439,4362,054-880-8242
75,95092439,4364,054-850-3292
76,95092439,4367,055-211-5753
77,95092439,4369,055-254-4173


In [14]:
import re
import pandas as pd

structure_results = []

for news_id in [95092486, 95092439]:
    text = new_df.loc[
        new_df["보도자료ID"].astype(str) == str(news_id),
        "본문"
    ].iloc[0]

    lines = text.splitlines()

    for index, line in enumerate(lines, start=1):
        value = line.strip()

        # 문서 구분에 사용하는 제목 후보 탐색
        if re.match(
            r"^(담당\s*부서|책임자|참고\s*\d+|붙임\s*\d*|별첨\s*\d*)$",
            value
        ):
            structure_results.append({
                "보도자료ID": news_id,
                "줄번호": index,
                "구분": value,
            })

structure_df = pd.DataFrame(structure_results)

display(structure_df)

,보도자료ID,줄번호,구분
0,95092486,24,담당 부서
1,95092486,26,책임자
2,95092486,35,담당 부서
3,95092486,37,책임자
4,95092439,467,참고 1
5,95092439,560,참고 2
6,95092439,572,참고 3
7,95092439,636,참고 4
8,95092439,729,참고 5
9,95092439,2310,참고 6


In [15]:
# 두 예외 자료의 본문 경계 확인
check_positions = {
    95092486: [24, 35],
    95092439: [467]
}

for news_id, positions in check_positions.items():
    text = new_df.loc[
        new_df["보도자료ID"].astype(str) == str(news_id),
        "본문"
    ].iloc[0]

    lines = text.splitlines()

    print(f"\n===== 보도자료 {news_id} =====")

    for position in positions:
        print(f"\n--- {position}행 주변 ---")

        for index in range(
            max(0, position - 5),
            min(len(lines), position + 3)
        ):
            print(f"{index + 1}행: {lines[index]}")


===== 보도자료 95092486 =====

--- 24행 주변 ---
20행: 나눌 예정이다. 앞으로 TF는 전문가 등의 발제를 통해 택시업 노동환경에 
21행: 대한 그간의 조사, 해외사례 등을 살펴보고, 노‧사의 의견을 충분히 수렴하
22행: 여 택시업 관련 실태조사 추진계획을 구체화할 예정이다.
23행: 붙임. 가칭｢택시업종 근로시간 등 노동환경 실태조사 TF｣ 킥오프 회의 개요
24행: 담당 부서
25행: 고용노동부
26행: 책임자
27행: 과  장 

--- 35행 주변 ---
31행: 사무관
32행: 정혜진(044-202-7616)
33행: 주무관
34행: 오혜란(044-202-7973)
35행: 담당 부서
36행: 국토교통부
37행: 책임자
38행: 과  장

===== 보도자료 95092439 =====

--- 467행 주변 ---
463행: 14. 교통정보 안내전화 및 인터넷 주소····································· 57
464행: 15. 교통불편신고 연락처································································ 58
465행: 
466행: - 15 -
467행: 참고 1
468행: 추석 연휴기간 통행실태조사 결과
469행: ※ 추석 연휴기간 통행실태 설문조사
470행:     (본 조사 8.12∼8.18, 한국교통연구원)


In [16]:
# 95092439의 본문 종료 및 참고자료 시작 부분 확인
text = new_df.loc[
    new_df["보도자료ID"] == 95092439,
    "본문"
].iloc[0]

lines = text.splitlines()

for index in range(430, min(470, len(lines))):
    print(f"{index + 1}행: {lines[index]}")

431행: ㅇ특별수송대책에 대한 홍보(안내방송 등)
432행: ㅇ시설물 안전관리 철저 
433행: 한국공항공사
434행: 인천국제공항공사
435행: ㅇ공항 이용객의 안전수송과 편의 도모
436행: ㅇ심야버스, 철도 운행 등 공항이용 서비스 강화
437행: ㅇ신속한 출입국 절차 지원
438행: 전국버스연합회
439행: 전국전세버스연합회 
440행: 한국자동차모빌리티
441행: 산업협회
442행: ㅇ고속도로 질서유지 단속 지원 
443행: ㅇ교통수단의 사전 안전점검·정비 실시 유도
444행: ㅇ종사원 안전교육 실시 및 서비스 제고 유도
445행: ㅇ안전운행 및 수송력 증강
446행: ㅇ자동차 제작사의 무상점검 서비스 시행
447행: 
448행: - 14 -
449행: 참 고 자 료
450행: 1. 추석 연휴기간 통행실태조사 결과······································· 15
451행: 2. 고속국도 일자별 예상교통량················································· 23
452행: 3. 주요도시간 최대 소요시간 예측··········································· 24
453행: 4. ‘25년도 고속국도 및 일반국도 개통 현황·························· 25
454행: 5. 고속국도 교통혼잡 예상구간 및 해소대책························ 27
455행: 6. 일반국도 교통혼잡 예상구간 및 해소대책························ 38
456행: 7. 고속국도 갓길차로 운영 현황··············································· 45
457행: 8. 고속국도 교통정보 서비스······················································ 48
458행: 9. 고속국도 정체시 국도 우회정보 제공 계획······················ 49
459행: 10.

In [17]:
from pathlib import Path
import pandas as pd

# 기존 정제 결과 복사
final_new_df = new_cleaned_df.copy()

# 95092486: 확인된 본문 23줄 적용
target_id = 95092486
mask = final_new_df["보도자료ID"].astype(str) == str(target_id)

original_text = final_new_df.loc[mask, "본문"].iloc[0]
article_text = "\n".join(original_text.splitlines()[:23]).strip()

final_new_df.loc[mask, "학습본문"] = article_text
final_new_df.loc[mask, "검수상태"] = "검수 필요"

# 95092439: 전체 원문 유지, 별도 검수
mask_long = final_new_df["보도자료ID"].astype(str) == "95092439"
final_new_df.loc[mask_long, "검수상태"] = "별도 검수 필요"

# 정제 후 글자 수 갱신
final_new_df["학습본문글자수"] = (
    final_new_df["학습본문"].str.len()
)

# 필요한 열만 선택
result_columns = [
    "보도자료ID", "제목", "담당부서", "등록일",
    "상세URL", "PDF_URL", "학습본문",
    "학습본문글자수", "검수상태"
]

result_df = final_new_df[result_columns].copy()

# 기존 파일 보호
output_file = Path("trafficnews_new_9_review.csv")

if output_file.exists():
    print("이미 파일이 있어 덮어쓰지 않았습니다.")
else:
    result_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print("검수용 CSV 저장 완료!")

saved_df = pd.read_csv(output_file)

print("저장 건수:", len(saved_df))
print("본문 누락:", saved_df["학습본문"].isna().sum())
print("저장 전후 본문 일치:",
      saved_df["학습본문"].tolist() == result_df["학습본문"].tolist())

display(saved_df[
    ["보도자료ID", "학습본문글자수", "검수상태"]
])

검수용 CSV 저장 완료!
저장 건수: 9
본문 누락: 0
저장 전후 본문 일치: True


,보도자료ID,학습본문글자수,검수상태
0,95092504,1509,검수 필요
1,95092486,680,검수 필요
2,95092459,2396,검수 필요
3,95092458,1409,검수 필요
4,95092447,1574,검수 필요
5,95092445,1629,검수 필요
6,95092439,47746,별도 검수 필요
7,95092438,1932,검수 필요
8,95092430,1376,검수 필요


In [18]:

from pathlib import Path
import pandas as pd

# 기존 10건과 신규 9건 불러오기
old_df = pd.read_csv("trafficnews_all_10_cleaned.csv")
new_df = pd.read_csv("trafficnews_new_9_review.csv")

# 기존 자료도 최종 학습 승인이 아닌 검수 대상으로 분류
old_df["검수상태"] = "검수 필요"

# 두 데이터 통합
all_review_df = pd.concat(
    [old_df, new_df],
    ignore_index=True
)

# 검증
if len(all_review_df) != 19:
    raise ValueError("통합 건수가 19건이 아닙니다.")

if all_review_df["보도자료ID"].duplicated().any():
    raise ValueError("중복 ID가 발견됐습니다.")

if all_review_df["학습본문"].isna().any():
    raise ValueError("학습본문 누락이 발견됐습니다.")

if all_review_df["학습본문"].str.strip().eq("").any():
    raise ValueError("빈 학습본문이 발견됐습니다.")

# 저장할 열 선택
columns = [
    "보도자료ID", "제목", "담당부서", "등록일",
    "상세URL", "PDF_URL", "학습본문", "검수상태"
]

all_review_df = all_review_df[columns]

# 기존 파일 보호
output_file = Path("trafficnews_all_19_review.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    all_review_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("통합 검수용 CSV 저장 완료!")

# 저장 결과 검증
saved = pd.read_csv(output_file)

print("저장 건수:", len(saved))
print("중복 ID:", saved["보도자료ID"].duplicated().sum())
print("본문 누락:", saved["학습본문"].isna().sum())
print(
    "본문 전체 일치:",
    saved["학습본문"].tolist() == all_review_df["학습본문"].tolist()
)
print("\n검수 상태:")
print(saved["검수상태"].value_counts())


통합 검수용 CSV 저장 완료!
저장 건수: 19
중복 ID: 0
본문 누락: 0
본문 전체 일치: True

검수 상태:
검수상태
검수 필요       18
별도 검수 필요     1
Name: count, dtype: int64


In [19]:
from pathlib import Path
import pandas as pd

# 통합 검수 데이터
df = pd.read_csv("trafficnews_all_19_review.csv")

# 기존에 리드를 작성한 보도자료 제외
review_df = df[
    df["보도자료ID"].astype(str) != "95092576"
].copy()

# 기사 리드 검수용 열 추가
review_df["리드"] = ""
review_df["판정"] = "검토대기"

# 기존 별도 검수 상태 유지
review_df.loc[
    review_df["검수상태"] == "별도 검수 필요",
    "판정"
] = "별도 검수 필요"

output_file = Path("trafficnews_lead_review.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    review_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("리드 검수용 CSV 저장 완료!")

saved = pd.read_csv(output_file)

print("검수 대상:", len(saved))
print("리드 작성 완료:", saved["리드"].notna().sum())
print("별도 검수 필요:",
      (saved["판정"] == "별도 검수 필요").sum())
print("저장 파일:", output_file)

리드 검수용 CSV 저장 완료!
검수 대상: 18
리드 작성 완료: 0
별도 검수 필요: 1
저장 파일: trafficnews_lead_review.csv


In [20]:

import re
import pandas as pd

# 원본 108건 불러오기
raw_df = pd.read_csv("trafficnews_all_108_raw.csv")

# 원본을 수정하지 않고 검사 결과만 별도 생성
quality_df = raw_df[
    ["보도자료ID", "제목", "페이지수"]
].copy()

# PDF 본문 상태 검사
quality_df["본문글자수"] = raw_df["본문"].str.len()

quality_df["전화번호포함"] = raw_df["본문"].str.contains(
    r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}",
    regex=True,
    na=False
)

quality_df["참고자료포함"] = raw_df["본문"].str.contains(
    r"(?m)^\s*참고\s*\d+",
    regex=True,
    na=False
)

quality_df["담당부서표기"] = raw_df["본문"].str.contains(
    r"담당\s*부서",
    regex=True,
    na=False
)

# 길이가 긴 문서는 별도 검토 대상으로 분류
quality_df["장문자료"] = (
    (quality_df["페이지수"] >= 10) |
    (quality_df["본문글자수"] >= 10000)
)

quality_df["검수상태"] = "일반 검수"

quality_df.loc[
    quality_df["장문자료"],
    "검수상태"
] = "장문 별도 검수"

print("===== 108건 품질 검사 =====")
print("전체:", len(quality_df))
print("전화번호 포함:", quality_df["전화번호포함"].sum())
print("참고자료 포함:", quality_df["참고자료포함"].sum())
print("담당부서 표기:", quality_df["담당부서표기"].sum())
print("장문 별도 검수:", quality_df["장문자료"].sum())

display(
    quality_df[
        ["보도자료ID", "페이지수", "본문글자수", "검수상태"]
    ]
)


===== 108건 품질 검사 =====
전체: 108
전화번호 포함: 108
참고자료 포함: 26
담당부서 표기: 102
장문 별도 검수: 5


,보도자료ID,페이지수,본문글자수,검수상태
0,95092576,2,1707,일반 검수
1,95092592,6,1844,일반 검수
2,95092593,2,1035,일반 검수
3,95092583,7,4065,일반 검수
4,95092574,3,2753,일반 검수
...,...,...,...,...
103,95091441,3,2568,일반 검수
104,95091427,2,1632,일반 검수
105,95091425,4,3309,일반 검수
106,95091414,3,1324,일반 검수


In [21]:
import re
import pandas as pd
from pathlib import Path

raw_df = pd.read_csv("trafficnews_all_108_raw.csv")

phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"
page_pattern = r"(?m)^[ \t]*-[ \t]*\d+[ \t]*-[ \t]*$"

def prepare_training_text(row):
    original = row["본문"]

    # 페이지 번호만 제거
    text = re.sub(page_pattern, "", original)
    lines = text.splitlines()

    # 담당부서가 독립된 한 줄인 경우 탐색
    positions = [
        i for i, line in enumerate(lines)
        if re.fullmatch(r"\s*담당\s*부서\s*", line)
    ]

    long_document = (
        row["페이지수"] >= 10 or len(original) >= 10000
    )

    # 경계가 불분명하면 원문 보존
    if len(positions) != 1 or long_document:
        return pd.Series([text, "별도 검수 필요"])

    article_lines = lines[:positions[0]]

    # 끝부분의 빈 줄과 단독 구분 기호만 정리
    while article_lines and article_lines[-1].strip() in {"", "-", "`"}:
        article_lines.pop()

    article_text = "\n".join(article_lines).strip()

    if not article_text or re.search(phone_pattern, article_text):
        return pd.Series([text, "별도 검수 필요"])

    return pd.Series([article_text, "검수 필요"])

# 새 데이터프레임에서 처리
prepared_df = raw_df.copy()

prepared_df[["학습본문", "검수상태"]] = prepared_df.apply(
    prepare_training_text,
    axis=1
)

prepared_df["학습본문글자수"] = prepared_df["학습본문"].str.len()

# 검수용 CSV 저장
output_file = Path("trafficnews_108_prepared.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    prepared_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("학습용 본문 후보 CSV 저장 완료!")

saved_df = pd.read_csv(output_file)

print("\n===== 자동 정제 결과 =====")
print("전체:", len(saved_df))
print(saved_df["검수상태"].value_counts())
print("본문 누락:", saved_df["학습본문"].isna().sum())
print("중복 ID:", saved_df["보도자료ID"].duplicated().sum())

학습용 본문 후보 CSV 저장 완료!

===== 자동 정제 결과 =====
전체: 108
검수상태
검수 필요       90
별도 검수 필요    18
Name: count, dtype: int64
본문 누락: 0
중복 ID: 0


In [22]:
import pandas as pd

df = pd.read_csv("trafficnews_108_prepared.csv")

# 자동 분리된 90건만 검사
targets = df[df["검수상태"] == "검수 필요"].copy()

# 정제한 본문이 원본에 그대로 포함되는지 확인
# 페이지 번호 제거로 인한 차이는 허용
def check_text_preservation(row):
    original = row["본문"]
    prepared = row["학습본문"]

    original_lines = [
        line for line in original.splitlines()
        if not line.strip().strip("- ").isdigit()
    ]

    prepared_lines = prepared.splitlines()

    # 원문 줄 순서와 내용 비교
    return all(
        line in original_lines
        for line in prepared_lines
        if line.strip()
    )

targets["원문보존검사"] = targets.apply(
    check_text_preservation,
    axis=1
)

print("===== 정제 결과 검사 =====")
print("검사 대상:", len(targets))
print("원문보존검사 통과:", targets["원문보존검사"].sum())
print("추가 확인 필요:", (~targets["원문보존검사"]).sum())

if (~targets["원문보존검사"]).any():
    display(
        targets.loc[
            ~targets["원문보존검사"],
            ["보도자료ID", "제목"]
        ]
    )

===== 정제 결과 검사 =====
검사 대상: 90
원문보존검사 통과: 79
추가 확인 필요: 11


,보도자료ID,제목
22,95092375,"[장관동정] 김윤덕 장관, SNS로 찾은 “따뜻한 버스기사” 직접 만나 표창"
27,95092266,모두의카드(기후동행패스) 서비스 9월 1일부터 시행
33,95092207,"김용석 대광위원장,인천 2호선 고양연장 예타현황 점검"
34,95092194,경기 5개 광역버스 노선 새로 달린다
35,95092191,"화물차 유가보조금, 인공지능(AI)으로 더 투명하게 관리한다"
43,95092130,"[장관동정] 김윤덕 장관, “주차장 진출입 방해 근절해 국민불편 해소할 것”"
64,95091901,"국가대표 교통카드 「모두의카드」, 이용자 500만 명 돌파"
70,95091819,"[차관동정] 홍지선 2차관, 고유가 극복을 위한 교통 · 물류업계 간담회 개최"
75,95091719,"[장관동정] 설 연휴, 국민 모두가 안전하고 편리하게 이동하도록 현장부터 점검합니다"
76,95091698,"[차관동정] 홍지선 2차관, 설 ‘택배 특별관리기간’ 맞아 택배 터미널 현장 점검"


In [23]:
import re
import pandas as pd

# 기존 검사에서 통과하지 못한 11건
failed_df = targets[
    ~targets["원문보존검사"]
].copy()

page_pattern = r"(?m)^[ \t]*-[ \t]*\d+[ \t]*-[ \t]*$"

results = []

for _, row in failed_df.iterrows():
    # 페이지 번호만 제거한 원본
    original = re.sub(page_pattern, "", row["본문"])
    prepared = row["학습본문"]

    # 공백 차이를 제외하고 비교
    original_compact = re.sub(r"\s+", "", original)
    prepared_compact = re.sub(r"\s+", "", prepared)

    # 정제본이 원본의 앞부분과 일치하는지 확인
    prefix_match = original_compact.startswith(prepared_compact)

    # 불일치 지점 확인
    difference_at = next(
        (
            i for i, (a, b) in enumerate(
                zip(original_compact, prepared_compact)
            )
            if a != b
        ),
        None
    )

    results.append({
        "보도자료ID": row["보도자료ID"],
        "공백제외_앞부분일치": prefix_match,
        "첫불일치위치": difference_at,
        "원본글자수": len(original_compact),
        "정제글자수": len(prepared_compact)
    })

result_df = pd.DataFrame(results)

print("검사 대상:", len(result_df))
print("앞부분 일치:", result_df["공백제외_앞부분일치"].sum())
print("추가 조사 필요:", (~result_df["공백제외_앞부분일치"]).sum())

display(result_df)

검사 대상: 11
앞부분 일치: 11
추가 조사 필요: 0


,보도자료ID,공백제외_앞부분일치,첫불일치위치,원본글자수,정제글자수
0,95092375,True,None,671,609
1,95092266,True,None,2873,1850
2,95092207,True,None,685,618
3,95092194,True,None,2717,1011
4,95092191,True,None,1455,1396
5,95092130,True,None,887,806
6,95091901,True,None,1413,1266
7,95091819,True,None,566,505
8,95091719,True,None,1312,1036
9,95091698,True,None,724,640


In [24]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("trafficnews_108_prepared.csv")

# 자동 분리된 본문 후보만 선택
lead_df = df[df["검수상태"] == "검수 필요"].copy()

# 리드 생성과 검수에 필요한 열 구성
lead_df = lead_df[
    ["보도자료ID", "제목", "등록일", "상세URL", "학습본문"]
].copy()

lead_df["리드"] = ""
lead_df["판정"] = "검토대기"

output_file = Path("trafficnews_90_lead_draft.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    lead_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print("리드 초안 작업 파일 생성 완료!")

saved = pd.read_csv(output_file)

print("리드 작성 대상:", len(saved))
print("작성 완료:", saved["리드"].notna().sum())
print("저장 파일:", output_file)

리드 초안 작업 파일 생성 완료!
리드 작성 대상: 90
작성 완료: 0
저장 파일: trafficnews_90_lead_draft.csv


In [25]:

import csv
import time
import requests
import pandas as pd
from pathlib import Path

input_file = Path("trafficnews_90_lead_draft.csv")
output_file = Path("trafficnews_90_lead_generated.csv")

df = pd.read_csv(input_file).fillna("")

columns = [
    "보도자료ID", "제목", "등록일",
    "상세URL", "학습본문", "리드", "판정"
]

# 실행을 다시 해도 이미 생성한 리드는 건너뛰기
completed_ids = set()

if output_file.exists():
    previous = pd.read_csv(output_file)
    completed_ids = set(previous["보도자료ID"].astype(str))

success_count = 0
failed_ids = []

for _, row in df.iterrows():
    news_id = str(row["보도자료ID"])

    if news_id in completed_ids:
        continue

    prompt = (
        "다음 보도자료를 바탕으로 한국어 뉴스기사의 "
        "첫 문장(리드)을 작성하세요.\n"
        "규칙:\n"
        "1. 한 문장만 작성하세요.\n"
        "2. 핵심 사실을 우선하세요.\n"
        "3. 원문에 없는 사실, 수치, 원인, 전망을 추가하지 마세요.\n"
        "4. 설명이나 제목 없이 리드만 출력하세요.\n\n"
        f"보도자료:\n{row['학습본문']}"
    )

    try:
        response = requests.post(
            "http://127.0.0.1:11434/api/generate",
            json={
                "model": "qwen2.5:3b",
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.2,
                    "num_ctx": 8192
                }
            },
            timeout=300
        )
        response.raise_for_status()

        lead = response.json()["response"].strip()

        if not lead:
            raise ValueError("빈 리드 생성")

        record = {
            "보도자료ID": news_id,
            "제목": row["제목"],
            "등록일": row["등록일"],
            "상세URL": row["상세URL"],
            "학습본문": row["학습본문"],
            "리드": lead,
            "판정": "검토대기"
        }

        new_file = not output_file.exists()

        with output_file.open(
            "a", newline="", encoding="utf-8-sig"
        ) as file:
            writer = csv.DictWriter(file, fieldnames=columns)

            if new_file:
                writer.writeheader()

            writer.writerow(record)

        completed_ids.add(news_id)
        success_count += 1

        print(f"완료: {len(completed_ids)}/{len(df)} - {news_id}")

    except (requests.RequestException, ValueError, KeyError) as error:
        failed_ids.append(news_id)
        print(f"오류: {news_id} - {error}")

    # 로컬 모델 호출 사이 짧은 간격
    time.sleep(0.2)

print("\n===== 리드 생성 결과 =====")
print("이번 실행 성공:", success_count)
print("실패:", len(failed_ids))

if output_file.exists():
    result = pd.read_csv(output_file)
    print("누적 생성:", len(result))
    print("검토대기:", (result["판정"] == "검토대기").sum())
    print("저장 파일:", output_file)


C:\Users\EZ\AppData\Local\Temp\ipykernel_5072\824294293.py:10: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  df = pd.read_csv(input_file).fillna("")


완료: 1/90 - 95092576
완료: 2/90 - 95092593
완료: 3/90 - 95092583
완료: 4/90 - 95092574
완료: 5/90 - 95092564
완료: 6/90 - 95092532
완료: 7/90 - 95092531
완료: 8/90 - 95092519
완료: 9/90 - 95092504
완료: 10/90 - 95092459
완료: 11/90 - 95092458
완료: 12/90 - 95092447
완료: 13/90 - 95092445
완료: 14/90 - 95092438
완료: 15/90 - 95092430
완료: 16/90 - 95092429
완료: 17/90 - 95092403
완료: 18/90 - 95092377
완료: 19/90 - 95092375
완료: 20/90 - 95092374
완료: 21/90 - 95092355
완료: 22/90 - 95092342
완료: 23/90 - 95092266
완료: 24/90 - 95092241
완료: 25/90 - 95092235
완료: 26/90 - 95092229
완료: 27/90 - 95092218
완료: 28/90 - 95092207
완료: 29/90 - 95092194
완료: 30/90 - 95092191
완료: 31/90 - 95092184
완료: 32/90 - 95092172
완료: 33/90 - 95092154
완료: 34/90 - 95092145
완료: 35/90 - 95092135
완료: 36/90 - 95092132
완료: 37/90 - 95092130
완료: 38/90 - 95092124
완료: 39/90 - 95092120
완료: 40/90 - 95092113
완료: 41/90 - 95092102
완료: 42/90 - 95092091
완료: 43/90 - 95092072
완료: 44/90 - 95092068
완료: 45/90 - 95092053
완료: 46/90 - 95092048
완료: 47/90 - 95092032
완료: 48/90 - 95092022
완

In [26]:

import re
import pandas as pd

df = pd.read_csv("trafficnews_90_lead_generated.csv")

def check_lead(row):
    source = str(row["학습본문"])
    lead = str(row["리드"])

    # 리드와 원문의 숫자 비교
    source_numbers = set(re.findall(r"\d+(?:[.,]\d+)*", source))
    lead_numbers = set(re.findall(r"\d+(?:[.,]\d+)*", lead))

    new_numbers = lead_numbers - source_numbers

    return pd.Series({
        "리드글자수": len(lead),
        "빈리드": not lead.strip(),
        "새로운수치": ", ".join(sorted(new_numbers)),
        "새로운수치의심": bool(new_numbers),
        "줄바꿈포함": "\n" in lead
    })

check_df = df.apply(check_lead, axis=1)

result_df = pd.concat(
    [df[["보도자료ID", "제목", "리드"]], check_df],
    axis=1
)

print("===== 리드 품질 검사 =====")
print("전체:", len(result_df))
print("빈 리드:", result_df["빈리드"].sum())
print("새로운 수치 의심:", result_df["새로운수치의심"].sum())
print("줄바꿈 포함:", result_df["줄바꿈포함"].sum())

display(result_df)


===== 리드 품질 검사 =====
전체: 90
빈 리드: 0
새로운 수치 의심: 2
줄바꿈 포함: 0


,보도자료ID,제목,리드,리드글자수,빈리드,새로운수치,새로운수치의심,줄바꿈포함
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,신도시 교통대책의 인허가 지연 요인과 현장 애로사항을 점검하는 특별점검 TF가 11...,57,False,,False,False
1,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",김용석 대광위원장은 12일 오후 고양창릉 공공주택지구 광역교통개선대책 사업 추진 상...,71,False,,False,False
2,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",정의경 국토교통부 제2차관은 11월 10일 오후 대장~홍대 광역철도 사업 현장에서 ...,119,False,,False,False
3,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",전남광주 지역 어르신들은 12월 7일부터 카드 한 장으로 도시철도 무료 이용과 교통...,60,False,,False,False
4,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",김용석 대광위원장은 하남교산 주요 교통시설의 적기 구축을 위해 광역교통개선대책 주요...,62,False,,False,False
...,...,...,...,...,...,...,...,...
85,95091441,"대광위, ‘K-도시철도’ 원 팀으로 베트남 협력의 새 문을 열다","대광위는 베트남 하노이를 방문해 한-베 도시철도 협력 사업을 공고화하고, 한국 기업...",72,False,,False,False
86,95091427,"공공기관 유휴부지, 생활물류시설로 활용",국토교통부가 도심 내 택배 물동량 증가에 따른 배송거리 증가와 교통혼잡을 해소하기 ...,84,False,,False,False
87,95091425,"신도시 교통대책 TF, 현장 협업 강화로 갈등사업 조정 성과",신도시 교통대책 TF가 현장 협업 강화로 인해 지자체와 관련 기관 간 갈등사업 조정...,59,False,,False,False
88,95091414,"[장관동정] 김윤덕 장관, “교통산업이 경쟁력을 높일 수 있도록 교통산업 혁신을 뒷...",김윤덕 국토교통부 장관은 14일 육운의 날 기념식에서 교통산업의 혁신을 통해 경쟁력...,82,False,,False,False


In [27]:
# 새로운 수치가 의심되는 2건만 확인
suspected_df = result_df[
    result_df["새로운수치의심"]
]

for _, row in suspected_df.iterrows():
    print("\n===== 수치 검증 대상 =====")
    print("보도자료 ID:", row["보도자료ID"])
    print("제목:", row["제목"])
    print("의심 수치:", row["새로운수치"])
    print("생성 리드:", row["리드"])

    original_text = df.loc[
        df["보도자료ID"] == row["보도자료ID"],
        "학습본문"
    ].iloc[0]

    print("\n===== 보도자료 원문 전체 =====")
    print(original_text)


===== 수치 검증 대상 =====
보도자료 ID: 95091656
제목: 우리 동네 교통문화, 얼마나 달라졌을까?
의심 수치: 2025
생성 리드: 2025년 교통문화지수는 전년 대비 0.61점 상승한 81.34점으로 나타났다.

===== 보도자료 원문 전체 =====
보도자료
보도시점 : 2026. 2. 1.(일) 11:00 이후(2. 2.(월) 조간) / 배포 : 2026. 1. 30.(금)
□ 국토교통부(장관 김윤덕)와 한국교통안전공단(이사장 정용식)이 전국 229개 
시‧군‧구를 대상으로 실시한 조사 결과, ’25년 교통문화지수* 는 81.34점
으로 전년(80.73점) 대비 0.61점 상승한 것으로 나타났다.
    * 국가승인통계로 매년 전국 229개 시·군·구(인구 30만 이상 시, 인구 30만 미만 시, 군, 구 
4개 그룹) 대상, 운전행태·보행행태·교통안전 항목에 대한 지표를 평가하여 지수화 
ㅇ 이는 운전자와 보행자의 교통법규 준수 수준이 전반적으로 개선되어, 국민의 
일상 속 교통안전 의식이 점차 자리 잡고 있음을 보여주는 결과다.
< 조사 항목별 결과 >
□ 전국 운전행태 조사 결과, 최근 5년간 대부분의 지표가 꾸준히 개선된 
것으로 나타났다. 특히 ▲ 이륜차 승차자 안전모 착용률(’2393.95%→’2597.16%), 
▲ 안전띠 착용 준수율(’2384.93%→’2585.38%), ▲ 음주운전 금지 준수율(’2396.78% → 
’2597.92%), ▲ 운전 중 스마트기기 미사용 준수율(’2361.07%→’2564.32%), ▲ 횡단
보도 정지선 준수율(’2381.82%→’2582.29%)은 최근 2년간 지속적인 상승세를 
보였다.
 ㅇ 다만, 신호준수율(’2496.80%→’2596.65%)은 여전히 높은 수준을 유지하고 
있으나, 전년 대비 소폭 하락해 지속적인 단속과 계도 활동이 필요한 
것으로 분석됐다.
□ 전국 보행행태 조사 결과, ▲횡단보도 신호 준수율(’2492.46%→’2592.80%), 
▲횡단 중 스마트기기 미

In [28]:
# 두 번째 수치 의심 자료만 확인
second = suspected_df.iloc[1]

print("보도자료 ID:", second["보도자료ID"])
print("제목:", second["제목"])
print("의심 수치:", second["새로운수치"])
print("생성 리드:", second["리드"])

# 원문에서 의심 수치와 관련된 줄 확인
source = df.loc[
    df["보도자료ID"] == second["보도자료ID"],
    "학습본문"
].iloc[0]

print("\n===== 의심 수치가 포함된 원문 줄 =====")

numbers = str(second["새로운수치"]).split(", ")

for line in source.splitlines():
    if any(number in line for number in numbers):
        print(line)

보도자료 ID: 95091493
제목: AI와 함께 그려가는 광역교통의 미래,「광역교통 R&D 로드맵(’26~’35)」
의심 수치: 2026
생성 리드: 2035년까지의 광역교통 분야 중장기 연구개발 방향을 제시하는 "광역교통 R&D 로드맵(2026~2035)" 최종 발표회가 12월 10일 서울에서 개최된다.

===== 의심 수치가 포함된 원문 줄 =====


In [29]:
# 두 번째 보도자료의 원문에서 핵심 표현 확인
news_id = 95091493

row = df.loc[
    df["보도자료ID"] == news_id
].iloc[0]

source = row["학습본문"]

keywords = [
    "로드맵",
    "2035",
    "’35",
    "'35",
    "12월 10일",
    "서울",
    "발표회",
]

print("===== 원문 핵심 표현 확인 =====")

for keyword in keywords:
    print(keyword, ":", keyword in source)

print("\n===== 원문 전체 =====")
print(source)

===== 원문 핵심 표현 확인 =====
로드맵 : True
2035 : True
’35 : True
'35 : False
12월 10일 : False
서울 : True
발표회 : True

===== 원문 전체 =====
보도자료
보도시점 : 2025. 12. 10.(수) 11:00 이후(12. 11.(목) 조간) / 배포 : 2025. 12. 10.(수)
AI와 함께 그려가는 광역교통의 미래,
｢광역교통 R&D 로드맵(’26~’35)｣ 
- 철도기술연구원 등 연구기관 합동 광역교통 분야 중장기 연구개발 전략 제시
□ 국토교통부(장관 김윤덕) 대도시권광역교통위원회(위원장 김용석, 이하 
대광위)는 2035년까지의 광역교통 분야 중장기 연구개발 방향을 제시하기 
위해 ｢광역교통 R&D 로드맵(’26~’35)｣(이하 로드맵) 최종 발표회*를 12월 
10일 오전 서울에서 개최한다.
    * 한국철도기술연구원, 한국건설기술연구원, 한국교통안전공단, 한국도로공사, 국가
철도공단, 한국철도공사, 국토교통과학기술진흥원 등 교통분야 전문가 약 100명 참석
□ 대광위는 그간 기술개발을 통해 출·퇴근시간 혼잡감소 등 광역교통 문제를 
해소하고자 Super-BRT*, 광역 콜버스**(M-DRT) 등 교통수단의 기술개발과 
함께 새로운 서비스를 도입하는 데 힘써왔다.
    * 기존 BRT보다 정시성·신속성·쾌적성·안정성 등을 향상한 고품질 BRT
   ** 이용자 수요에 따라 운행 경로·시간·횟수를 가변적으로 운행하는 광역 버스
□ 이번 로드맵은 광역교통 분야 연구개발에 인공지능(AI) 등 첨단기술을 
적극 도입하고, 기후위기와 각종 재난으로 높아진 국민의 안전에 대한 
요구에 부응하기 위해 철도, 도로, AI 등 각 분야 전문가들과 함께 논의
하여 총 21개 핵심기술을 선정하였다. 주요 추진과제는 다음과 같다.
 ㅇ (교통서비스 확대) 혼잡한 출·퇴근 시간 승·하차시간을 획기적으로 단축
할 수 있는 태그리스* 시스템을 고도화하고, AI·디지털 트윈을 활용한 
철도·환승

In [30]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("trafficnews_90_lead_generated.csv")

# 검수할 항목 추가
df["수치검토"] = "검토대기"
df["날짜기관검토"] = "검토대기"
df["사실관계검토"] = "검토대기"
df["최종판정"] = "검토대기"
df["수정리드"] = ""
df["검토의견"] = ""

# 자동 검사에서 확인한 2건의 수치 결과만 반영
checked_ids = [95091656, 95091493]

df.loc[
    df["보도자료ID"].isin(checked_ids),
    "수치검토"
] = "기초검사 통과"

output_file = Path("trafficnews_90_lead_review.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print("리드 검수용 CSV 생성 완료!")

saved = pd.read_csv(output_file)

print("전체:", len(saved))
print("최종 검토대기:", (saved["최종판정"] == "검토대기").sum())
print("기초 수치검사 통과:", (saved["수치검토"] == "기초검사 통과").sum())
print("저장 파일:", output_file)

리드 검수용 CSV 생성 완료!
전체: 90
최종 검토대기: 90
기초 수치검사 통과: 2
저장 파일: trafficnews_90_lead_review.csv


In [1]:
import pandas as pd

df_prepared = pd.read_csv("trafficnews_108_prepared.csv")

print("전체 자료:", len(df_prepared))
print("\n컬럼 목록:")
print(df_prepared.columns.tolist())

print("\n검수 상태별 건수:")
for column in df_prepared.columns:
    if "검수" in column or "판정" in column:
        print(f"\n[{column}]")
        print(df_prepared[column].value_counts(dropna=False))

전체 자료: 108

컬럼 목록:
['제목', '담당부서', '등록일', '상세URL', 'PDF_URL', '본문', '페이지수', '본문글자수', '보도자료ID', '학습본문', '검수상태', '학습본문글자수']

검수 상태별 건수:

[검수상태]
검수상태
검수 필요       90
별도 검수 필요    18
Name: count, dtype: int64


In [2]:
# 별도 검수가 필요한 18건만 추출
extra_df = df_prepared[
    df_prepared["검수상태"] == "별도 검수 필요"
].copy()

# 본문 길이와 학습본문 길이 비교
extra_df["본문길이"] = extra_df["본문"].fillna("").str.len()
extra_df["학습본문길이"] = extra_df["학습본문"].fillna("").str.len()

display(
    extra_df[
        ["보도자료ID", "제목", "페이지수", "본문길이", "학습본문길이"]
    ].sort_values("학습본문길이")
)

,보도자료ID,제목,페이지수,본문길이,학습본문길이
86,95091595,"[차관동정] 홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검",2,1101,1091
11,95092486,"택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다",2,1205,1205
6,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",2,1460,1450
52,95092050,외래객 지역 이동 편의 높여 방한 관광 활성화한다,2,1510,1500
97,95091491,국민·전문가 의견 듣는다 … 10일 「제6차 국가물류기본계획」 공청회,2,1659,1649
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6,1844,1814
56,95091996,"화물차, 버스 유가연동보조금 지급한도 리터당 최대 280원으로 53％ 상향",3,1884,1869
36,95092189,"도시철도 계획부터 건설까지,대광위가 ‘원스톱’으로 지원",3,1921,1906
71,95091815,화물차 불법운행 합동단속… 도로안전 강화,3,2267,2252
90,95091543,"김용석 대광위원장, 광주·전남 지역거점 광역교통망 논의… 청년 K-패스 현장 의견...",3,2350,2335


In [3]:
import pandas as pd
import json
import urllib.request

# 기존 승인 자료와 중복되지 않는 후보 12건 선정
approved_df = pd.read_csv("trafficnews_90_lead_review_approved.csv")

candidate_df = (
    extra_df[
        ~extra_df["보도자료ID"].isin(approved_df["보도자료ID"])
    ]
    .sort_values("학습본문길이")
    .head(12)
    .copy()
)

generated_leads = []

for _, row in candidate_df.iterrows():
    source = str(row["학습본문"])

    prompt = (
        "다음 국토교통부 보도자료를 읽고 뉴스기사 첫 문장(리드)을 "
        "한국어 한 문장으로 작성하세요.\n"
        "원문에 없는 날짜, 수치, 기관명, 원인, 전망을 추가하지 마세요.\n"
        "기사 첫 문장만 출력하세요.\n\n"
        f"보도자료:\n{source}"
    )

    request_data = {
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2,
            "num_ctx": 8192
        }
    }

    try:
        request = urllib.request.Request(
            "http://127.0.0.1:11434/api/generate",
            data=json.dumps(request_data).encode("utf-8"),
            headers={"Content-Type": "application/json"}
        )

        with urllib.request.urlopen(request, timeout=180) as response:
            result = json.load(response)

        lead = result.get("response", "").strip()
        status = "생성완료" if lead else "생성실패"

    except Exception as error:
        lead = ""
        status = f"오류: {error}"

    generated_leads.append(lead)

    print(f"{row['보도자료ID']} | {status}")
    print("제목:", row["제목"])
    print("리드:", lead)
    print("-" * 60)

candidate_df["리드"] = generated_leads
candidate_df["최종판정"] = "검토대기"

print("\n생성 성공:", candidate_df["리드"].str.strip().ne("").sum())
print("총 후보:", len(candidate_df))

95091595 | 생성완료
제목: [차관동정] 홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검
리드: 2026년 1월 7일, 홍지선 국토교통부 제2차관은 취임 이후 첫 현장 방문으로 경기 광주시 곤지암읍에 위치한 쿠팡CFS 곤지암1센터를 찾아 물류센터의 시설 관리와 종사자 근로 여건을 점검했다.
------------------------------------------------------------
95092486 | 생성완료
제목: 택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다
리드: 2026년 10월 8일(목) 오전 9시, 고용노동부와 국토교통부는 서울 직업능력심사평가원에서 택시업종 노동환경 실태를 조사하기 위한 가칭 '택시업종 근로시간 등 노동환경 실태조사 TF' 킥오프 회의를 개최했다. 이 회의는 노동계, 경영계, 정부 등이 참여해 향후 검토사항, TF 운영 일정 등을 논의하는 자리였다.
------------------------------------------------------------
95092547 | 생성완료
제목: 문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.
리드: 문체부와 국토부는 11월 2일, 정부 세종청사에서 '제3회 관광-교통 정책협의회'를 개최하고, 외래관광객 3천만 시대를 향한 범부처 원팀 협업체계를 확립했다.
------------------------------------------------------------
95092050 | 생성완료
제목: 외래객 지역 이동 편의 높여 방한 관광 활성화한다
리드: 2026년 5월 27일, 문화체육관광부와 국토교통부는 '관광-교통 정책협의회'를 통해 외래객 3천만 명 시대를 이끌기 위한 방안을 논의했다.
------------------------------------------------------------
95091491 | 생성완료
제목: 국민·전문가 의견 듣는다 … 10일 「제6차 국가물류기본계획」 공청회
리드: 국토교통부와

In [4]:
for _, row in candidate_df.iterrows():
    print("=" * 70)
    print("ID:", row["보도자료ID"])
    print("제목:", row["제목"])
    print("AI 리드:", row["리드"])
    print("원문 앞부분:")
    print(str(row["학습본문"])[:1800])
    print()

ID: 95091595
제목: [차관동정] 홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검
AI 리드: 2026년 1월 7일, 홍지선 국토교통부 제2차관은 취임 이후 첫 현장 방문으로 경기 광주시 곤지암읍에 위치한 쿠팡CFS 곤지암1센터를 찾아 물류센터의 시설 관리와 종사자 근로 여건을 점검했다.
원문 앞부분:

동정자료
보도시점 : 배포 즉시 보도 가능 / 배포 : 2026. 1. 7.(수)
홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검
-경기 광주시 위치한 물류센터 찾아, 시설 관리 및 종사자 근로여건 점검-
□ 홍지선 국토교통부 제2차관은 1월 7일 오전 취임 이후 첫 현장 행보로 
경기 광주시 곤지암읍에 위치한 쿠팡CFS 곤지암1센터*를 찾아 물류
센터 시설과 종사자의 근로 여건을 점검했다.
    * (면적) 대지 8,485㎡(지하1층 지상1~4층), (보관물품) 신선제품, (종사인원) 약 900여명
 ㅇ 이번 점검은 최근 쿠팡 관련 국회 연석 청문회 등을 통해 제기된 쿠팡 
물류센터의 안전관리실태와 근로 여건 전반에 대한 우려를 직접 확인
하고, 물류시설 운영 과정에서 관계 법령과 안전관리 규정이 제대로 
준수되고 있는지를 점검하기 위해 마련됐다.
□ 물류센터를 찾은 홍 차관은 쿠팡CFS로부터 물류센터 운영과 종사자 
근무 현황 등을 보고 받고,
 ㅇ “국민 생활과 직결된 물류시설은 무엇보다 안전이 최우선되어야 
한다”며, “관계 법령과 안전관리 기준이 현장에서 제대로 이행되고 
있는지를 꼼꼼하게 살펴보겠다”고 밝혔다.
 ㅇ 특히, “이번 청문회에서 쿠팡의 물류 자회사인 쿠팡CFS와 쿠팡CLS가 
운영하는 물류센터에 대한 다양한 문제가 제기되었다”면서, “우리부는 
지자체 등 관계 기관과 함께 철저히 조사하여 위법 사항 확인 시 법령에 
따라 엄중하게 조치할 것”이라고 밝혔다.
□ 이어, 홍 차관은 물류센터 내 종사자들의 근로 현장을 점검한 뒤, “연초
부터 수고해주시는 근무자들의 노고에 감사하며, 정부도 종사자가 안전

In [5]:
for _, row in candidate_df.iterrows():
    print(f"\nID: {row['보도자료ID']}")
    print(f"제목: {row['제목']}")
    print(f"AI 리드: {row['리드']}")
    print(f"원문 핵심 부분: {str(row['학습본문'])[:500]}")
    print("-" * 40)


ID: 95091595
제목: [차관동정] 홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검
AI 리드: 2026년 1월 7일, 홍지선 국토교통부 제2차관은 취임 이후 첫 현장 방문으로 경기 광주시 곤지암읍에 위치한 쿠팡CFS 곤지암1센터를 찾아 물류센터의 시설 관리와 종사자 근로 여건을 점검했다.
원문 핵심 부분: 
동정자료
보도시점 : 배포 즉시 보도 가능 / 배포 : 2026. 1. 7.(수)
홍지선 2차관, 취임 첫 현장 행보로 쿠팡 물류센터 시설 점검
-경기 광주시 위치한 물류센터 찾아, 시설 관리 및 종사자 근로여건 점검-
□ 홍지선 국토교통부 제2차관은 1월 7일 오전 취임 이후 첫 현장 행보로 
경기 광주시 곤지암읍에 위치한 쿠팡CFS 곤지암1센터*를 찾아 물류
센터 시설과 종사자의 근로 여건을 점검했다.
    * (면적) 대지 8,485㎡(지하1층 지상1~4층), (보관물품) 신선제품, (종사인원) 약 900여명
 ㅇ 이번 점검은 최근 쿠팡 관련 국회 연석 청문회 등을 통해 제기된 쿠팡 
물류센터의 안전관리실태와 근로 여건 전반에 대한 우려를 직접 확인
하고, 물류시설 운영 과정에서 관계 법령과 안전관리 규정이 제대로 
준수되고 있는지를 점검하기 위해 마련됐다.
□ 물류센터를 찾은 홍 차관은 쿠팡CFS로부터 물류센터 운영과 종사자 
근무 현황 등을 보고 받고,
 ㅇ “국민 생활
----------------------------------------

ID: 95092486
제목: 택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다
AI 리드: 2026년 10월 8일(목) 오전 9시, 고용노동부와 국토교통부는 서울 직업능력심사평가원에서 택시업종 노동환경 실태를 조사하기 위한 가칭 '택시업종 근로시간 등 노동환경 실태조사 TF' 킥오프 회의를 개최했다. 이 회의는 노동계, 경영계, 정부 등이 참여해 향후 검토사항, TF 운영 일정 등을 논의하는 자리였다.
원문 핵심 부분: 고용
노동부
보도자료
보도시점
2026. 10. 8.

In [6]:
candidate_df.to_csv(
    "trafficnews_extra_12_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("추가 12건 검수 파일 저장 완료")
print("저장 건수:", len(candidate_df))

추가 12건 검수 파일 저장 완료
저장 건수: 12
